In [ ]:
from pathlib import Path
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

from setup_utils import create_minio_spark_session, DEV_LOCATION_ID_LIST

#### Start the spark session

In [ ]:
spark = create_minio_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

#### Initialize a TEEHR Evaluation with read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

#### Now you can pull in a subset of data from the TEEHR-Cloud warehouse via the API
Note: If you encounter a time out error, try reducing the page_size arg

Download domain variables, location data, and attribute data

In [ ]:
print("Loading configurations.")
ev.download.configurations(load=True)

print("Loading units.")
ev.download.units(load=True)

print("Loading variables.")
ev.download.variables(load=True)

print("Loading locations.")
ev.download.locations(
    ids=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location croswalks.")
ev.download.location_crosswalks(
    primary_location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location attributes.")
ev.download.attributes(load=True)
ev.download.location_attributes(
    location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading domain variables, locations, crosswalks, and attributes complete!")

Download forecast timeseries and obs

In [ ]:
ev.download.primary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="usgs_observations",
    variable_name="streamflow_hourly_inst",
    start_date="2026-05-01",
    end_date="2026-05-12",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_medium_range",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-02",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_hawaii",
    variable_name="streamflow_15min_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_alaska",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_puertorico",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)

Download retrospective simulation timeseries and obs

In [ ]:
ev.download.primary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="usgs_observations",
    variable_name="streamflow_hourly_inst",
    start_date="2020-05-01",
    end_date="2020-05-31",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_retrospective",
    variable_name="streamflow_hourly_inst",
    start_date="2020-05-01",
    end_date="2020-05-31",
    load=True,
)

To run the NWM forcing workflows you need to add the basin pixel weights table manually at this time

In [ ]:
# Load the pre-created weights table from local data
df = pd.read_parquet("local_data/dev_grid_pixel_coverage_weights.parquet")
ev.table("grid_pixel_coverage_weights").load_dataframe(
    df=df,
    write_mode="create_or_replace"
)
# Download the corresponding USGS basin locations
ev.download.locations(
    ids=df.location_id.unique().tolist(),
    load=True
)

Now download HUC6 basins for the data management dashboard

In [ ]:
ev.download.locations(prefix="huc6", load=True)

In [ ]:
spark.stop()